In [1]:
! pip install setfit datasets scikit-learn pandas "transformers<5.0"

In [2]:
import setfit
import pickle

import pandas as pd
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

ImportError: cannot import name 'default_logdir' from 'transformers.training_args' (/usr/local/lib/python3.12/dist-packages/transformers/training_args.py)

In [ ]:
import pickle

import pandas as pd
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

with open("clustered_by_label.pkl", "rb") as f:
    clustered_by_label = pickle.load(f)

rows = []
for label, payload in clustered_by_label.items():
    for text in payload.get("representative_texts", []):
        rows.append({"text": str(text), "label_raw": str(label)})

rep_df = pd.DataFrame(rows).drop_duplicates(subset=["text"]).reset_index(drop=True)
print({"representative_rows": len(rep_df), "raw_labels": rep_df["label_raw"].value_counts().to_dict()})
rep_df.head()


{'representative_rows': 100, 'raw_labels': {'casual': 20, 'possibly_needs_caution': 20, 'probably_needs_caution': 20, 'needs_caution': 20, 'needs_intervention': 20}}


,text,label_raw
0,"Yeah, but I feel bad about how terrible I am.",casual
1,She has always treated me unfairly.,casual
2,"I need to tell them not to come next time, I t...",casual
3,I don't always put kitchen utensils in the exa...,casual
4,He could still come back to a relationship wit...,casual


In [ ]:
rep_df["label_3"] = rep_df["label_raw"].replace(
    {
        "casual": "safe",
        "possibly_needs_caution": "possibly_needs_caution",
        "probably_needs_caution": "probably_needs_caution",
        "needs_caution": "needs_caution",
        "needs_intervention": "needs_intervention",
    }
)

label_encoder = LabelEncoder()
rep_df["label_enc"] = label_encoder.fit_transform(rep_df["label_3"])

train_df_setfit, valid_df_setfit = train_test_split(
    rep_df,
    test_size=0.30,
    random_state=42,
    stratify=rep_df["label_enc"],
)

print({"train_rows": len(train_df_setfit), "valid_rows": len(valid_df_setfit), "classes": label_encoder.classes_.tolist()})
valid_df_setfit.head(2)


{'train_rows': 70, 'valid_rows': 30, 'classes': ['needs_caution', 'needs_intervention', 'possibly_needs_caution', 'probably_needs_caution', 'safe']}


,text,label_raw,label_3,label_enc
23,I'm going to end up married to her.,possibly_needs_caution,possibly_needs_caution,2
39,"I like looking at pretty things, when it comes...",possibly_needs_caution,possibly_needs_caution,2


In [ ]:
train_dataset = Dataset.from_pandas(
    train_df_setfit[["text", "label_enc"]].rename(columns={"label_enc": "label"}),
    preserve_index=False,
)

model_id = "sentence-transformers/all-MiniLM-L6-v2"
model = SetFitModel.from_pretrained(model_id, labels=[0, 1, 2])

args = TrainingArguments(
    batch_size=16,
    num_epochs=3,
    evaluation_strategy="no",
    save_strategy="no",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

trainer.train()
print({"model": model_id, "epochs": 3})


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
The `evaluation_strategy` argument is deprecated and will be removed in a future version. Please use `eval_strategy` instead.


Map:   0%|          | 0/70 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 3920
  Batch size = 16
  Num epochs = 3
d:\DevTools\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,0.603400
50,0.291700
100,0.236100
150,0.231500
200,0.203600
250,0.149700
300,0.072500
350,0.039800
400,0.022200
450,0.013200


{'model': 'sentence-transformers/all-MiniLM-L6-v2', 'epochs': 3}


In [ ]:
test = valid_df_setfit.copy()
texts = test["text"].tolist()

pred_enc = [int(pred) for pred in model.predict(texts, use_labels=False)]
probas = model.predict_proba(texts)
pred_labels = label_encoder.inverse_transform(pred_enc)

out = pd.DataFrame(
    {
        "text": texts,
        "true": test["label_3"].tolist(),
        "pred": pred_labels.tolist(),
        "true=pred": (test["label_3"] == pred_labels).tolist(),
    }
)
proba_df = pd.DataFrame(probas, columns=[f"p_{c}" for c in label_encoder.classes_])
print(pd.concat([out, proba_df], axis=1).to_string(index=False))


                                                                                                 text                   true                   pred  true=pred  p_needs_caution  p_needs_intervention  p_possibly_needs_caution  p_probably_needs_caution   p_safe
                                                                  I'm going to end up married to her. possibly_needs_caution probably_needs_caution      False         0.094356              0.044111                  0.142661                  0.584108 0.134764
           I like looking at pretty things, when it comes to women that's the only thing that matters possibly_needs_caution          needs_caution      False         0.414105              0.185444                  0.119018                  0.251710 0.029723
                                      I fell in love with my best friend's ex and lost all my friends probably_needs_caution possibly_needs_caution      False         0.064323              0.095773                  0.511387

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Add confidence (max probability) and prediction correctness
out['confidence'] = proba_df.max(axis=1).values
out['is_correct'] = out['true'] == out['pred']

# Create confidence categories
out['confidence_category'] = pd.cut(
    out['confidence'],
    bins=[0, 0.5, 0.7, 1.0],
    labels=['low', 'medium', 'high']
)

# Create confusion categories
def categorize(row):
    if row['is_correct']:
        return f"correct_{row['confidence_category']}"
    else:
        return f"wrong_{row['confidence_category']}"

out['category'] = out.apply(categorize, axis=1)

# Display summary
print("Category Distribution:")
print(out['category'].value_counts().sort_index())
print("\n" + "="*80 + "\n")

# Show examples from each category
for cat in sorted(out['category'].unique()):
    subset = out[out['category'] == cat][['text', 'true', 'pred', 'confidence']]
    print(f"\n{cat.upper()} (n={len(subset)}):")
    print(subset.head(3).to_string(index=False))

# Classification metrics
print("\n" + "="*80 + "\n")
print("Overall Metrics:")
print(classification_report(out['true'], out['pred']))

Category Distribution:
category
correct_low        2
correct_medium     4
wrong_high         3
wrong_low         14
wrong_medium       7
Name: count, dtype: int64



CORRECT_LOW (n=2):
                                                                   text                   true                   pred  confidence
Yes, they have been being so hard to manage in a class for a few weeks. possibly_needs_caution possibly_needs_caution    0.493932
             I'm planning to put the woman asleep then molest her cunt.     needs_intervention     needs_intervention    0.412351

CORRECT_MEDIUM (n=4):
                                           text               true               pred  confidence
            I was busy running from the police. needs_intervention needs_intervention    0.524424
It's all right, it's written in an ironic tone.      needs_caution      needs_caution    0.651604
    No, I don't have any problem with that guy.               safe               safe    0.685509

WRONG_HIG

In [ ]:
import pickle
# pickle set fit model
with open("setfit_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [8]:
from datasets import load_dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import torch

# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

base_model = "sentence-transformers/all-MiniLM-L6-v2"

snli = load_dataset("snli", split="train")
label_names = list(snli.features["label"].names)
label_ids = list(range(len(label_names)))

split = snli.train_test_split(test_size=50, seed=42, stratify_by_column="label")
test_raw = split["test"]
pool_raw = split["train"]

def add_text(example):
    return {
        "text": f"Premise: {example['premise']}\nHypothesis: {example['hypothesis']}"
    }

def to_setfit_dataset(dataset):
    return dataset.map(
        add_text,
        remove_columns=[column for column in dataset.column_names if column != "label"],
    )

test_ds = to_setfit_dataset(test_raw)

def run_experiment(train_size, seed):
    train_raw = pool_raw.train_test_split(
        train_size=train_size,
        seed=seed,
        stratify_by_column="label",
    )["train"]
    train_ds = to_setfit_dataset(train_raw)

    # Explicitly load model to device
    print(f"Loading model {base_model} to {device}...")
    model = SetFitModel.from_pretrained(base_model, labels=label_ids).to(device)

    # Using max_steps to cap training time if the dataset is large
    args = TrainingArguments(
        batch_size=16,
        num_epochs=1,
        max_steps=10000, # Limit steps for speed
        evaluation_strategy="no",
        save_strategy="no",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
    )

    print("Starting training...")
    trainer.train()
    print("Training complete.")

    true_ids = list(test_ds["label"])
    test_texts = list(test_ds["text"])
    pred_ids = [int(pred) for pred in model.predict(test_texts, use_labels=False)]
    true_labels = [label_names[int(label)] for label in true_ids]
    pred_labels = [label_names[int(label)] for label in pred_ids]

    results = pd.DataFrame(
        {
            "true": true_labels,
            "pred": pred_labels,
            "correct": [true == pred for true, pred in zip(true_labels, pred_labels)],
        }
    )

    return results

# Run a quick test
# macro scores: 0.1->0.35->0.6->0.8 for samples 10,30,100,1000 (10k steps)
experiment_results = {}
for train_size in [1000]:
    experiment_results[train_size] = run_experiment(train_size, seed=42)

Using device: cuda
Loading model sentence-transformers/all-MiniLM-L6-v2 to cuda...


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
The `evaluation_strategy` argument is deprecated and will be removed in a future version. Please use `eval_strategy` instead.


Starting training...


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
***** Running training *****
  Num unique pairs = 160000
  Batch size = 16
  Num epochs = 1
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/na

Step,Training Loss
1,0.419200
50,0.293000
100,0.260200
150,0.258500
200,0.256900
250,0.254600
300,0.255200
350,0.251900
400,0.255500
450,0.256600


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-package

Training complete.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


In [11]:
print(classification_report(experiment_results[1000]['true'], experiment_results[1000]['pred']))

               precision    recall  f1-score   support

contradiction       0.86      0.71      0.77        17
   entailment       0.82      0.82      0.82        17
      neutral       0.68      0.81      0.74        16

     accuracy                           0.78        50
    macro avg       0.79      0.78      0.78        50
 weighted avg       0.79      0.78      0.78        50



/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [20]:
from datasets import load_dataset
from setfit import SetFitModel, Trainer, TrainingArguments
import pandas as pd
import torch

# Use detected device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load banking77 dataset
banking_ds = load_dataset("banking77", split="train")
banking_test_ds = load_dataset("banking77", split="test")

# Get label info
label_names = banking_ds.features["label"].names
label_ids = list(range(len(label_names)))

def run_banking_experiment(train_size_per_label=8):
    # SetFit works best with few-shot, so we'll sample per label
    train_ds = banking_ds.shuffle(seed=42).select(range(train_size_per_label * len(label_names)))

    print(f"Loading model for Banking77 to {device}...")
    model = SetFitModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", labels=label_ids).to(device)

    args = TrainingArguments(
        batch_size=16,
        num_epochs=3,
        max_steps=5000, # Cap for speed
        evaluation_strategy="no",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds
    )

    print("Starting Banking77 training...")
    trainer.train()
    print("Training complete.")

    # Evaluate on a small subset of test for speed
    sample_test = banking_test_ds.shuffle(seed=42).select(range(100))
    metrics = trainer.evaluate(sample_test)
    print(f"Evaluation Metrics: {metrics}")

    return model

banking_model = run_banking_experiment(train_size_per_label=25)

Using device: cuda
Loading model for Banking77 to cuda...


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
The `evaluation_strategy` argument is deprecated and will be removed in a future version. Please use `eval_strategy` instead.
/usr/local/lib/python3.

Map:   0%|          | 0/1925 [00:00<?, ? examples/s]

Starting Banking77 training...


***** Running training *****
  Num unique pairs = 80000
  Batch size = 16
  Num epochs = 3
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/jupyter_client/sessi

Training complete.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


Evaluation Metrics: {'accuracy': 0.72}


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [21]:
from sklearn.metrics import classification_report
import numpy as np

# Evaluate on a larger sample for a meaningful report
eval_sample = banking_test_ds.shuffle(seed=42).select(range(1000))

# Get predictions
print("Generating predictions for evaluation...")
preds = banking_model.predict(eval_sample["text"], use_labels=False)
y_pred = [int(p) for p in preds]
y_true = eval_sample["label"]

# Identify which labels are actually present in the true/pred set to avoid the mismatch error
# Alternatively, we pass all known label_ids to ensure the report matches target_names
present_labels = sorted(list(set(y_true) | set(y_pred)))
filtered_target_names = [label_names[i] for i in present_labels]

# Print report
print("\nBanking77 Classification Report (Sample):")
print(classification_report(y_true, y_pred, labels=present_labels, target_names=filtered_target_names))

Generating predictions for evaluation...

Banking77 Classification Report (Sample):
                                                  precision    recall  f1-score   support

                                activate_my_card       1.00      0.88      0.93        16
                                       age_limit       0.94      1.00      0.97        16
                         apple_pay_or_google_pay       1.00      1.00      1.00        20
                                     atm_support       1.00      1.00      1.00        11
                                automatic_top_up       1.00      1.00      1.00        10
         balance_not_updated_after_bank_transfer       0.41      0.82      0.55        11
balance_not_updated_after_cheque_or_cash_deposit       0.86      0.95      0.90        19
                         beneficiary_not_allowed       0.50      0.80      0.62        15
                                 cancel_transfer       1.00      1.00      1.00        13
               

/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=ut

In [19]:
from sklearn.metrics import classification_report
import numpy as np

# Evaluate on a larger sample for a meaningful report
eval_sample = banking_test_ds.shuffle(seed=42).select(range(1000))

# Get predictions
print("Generating predictions for evaluation...")
preds = banking_model.predict(eval_sample["text"], use_labels=False)
y_pred = [int(p) for p in preds]
y_true = eval_sample["label"]

# Identify which labels are actually present in the true/pred set to avoid the mismatch error
# Alternatively, we pass all known label_ids to ensure the report matches target_names
present_labels = sorted(list(set(y_true) | set(y_pred)))
filtered_target_names = [label_names[i] for i in present_labels]

# Print report
print("\nBanking77 Classification Report (Sample):")
print(classification_report(y_true, y_pred, labels=present_labels, target_names=filtered_target_names))

Generating predictions for evaluation...

Banking77 Classification Report (Sample):


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/

                                                  precision    recall  f1-score   support

                                activate_my_card       1.00      0.88      0.93        16
                                       age_limit       0.94      1.00      0.97        16
                         apple_pay_or_google_pay       0.75      0.45      0.56        20
                                     atm_support       0.91      0.91      0.91        11
                                automatic_top_up       1.00      0.70      0.82        10
         balance_not_updated_after_bank_transfer       0.44      0.64      0.52        11
balance_not_updated_after_cheque_or_cash_deposit       0.68      1.00      0.81        19
                         beneficiary_not_allowed       0.50      0.60      0.55        15
                                 cancel_transfer       1.00      1.00      1.00        13
                            card_about_to_expire       0.75      1.00      0.86        18
         

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12